In [ ]:
import pyreadstat
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

ABEL

In [ ]:
df, meta = pyreadstat.read_sav("T0 pats pre-interventie 20140528 origineel.sav")

In [ ]:
df = df[['Trialnr','opl']]
new_df = df.rename(columns={"Trialnr": "filename"})
new_df["filename"] = new_df["filename"].astype(int).astype(str)
new_df = new_df[new_df["opl"]< 10]

Correlation education level and text metrics (Corresponding to thesis section "Correlation text metrics with patient education level")

In [ ]:
level_df = pd.read_csv("Level_Analysis/Abel_Level_Scores.csv")
level_df = level_df[level_df["label"]=="Patient"]
level_df['filename'] = level_df['filename'].str.replace('_A', '').str.replace(' AB', '')
level_df

In [ ]:
counts = new_df['opl'].value_counts().sort_index()

# force categories 1–9 to exist
counts = counts.reindex(range(1, 10), fill_value=0)

fig, ax = plt.subplots()
counts.plot(kind='bar', ax=ax)

ax.set_xlabel("Education level")
ax.set_ylabel("Number of patients")
ax.set_xticklabels(counts.index.astype(str), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
merged_df = new_df.merge(level_df, how = 'inner', on = ['filename'])
merged_df = merged_df.drop(columns=['Unnamed: 0', 'label'])

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("abel_education.csv")

In [ ]:
metrics = ['avg_sent_length', 'avg_syllables', 'fres', 'pass_pct', 'ttr']

In [ ]:


for m in metrics:
    rho, p = spearmanr(merged_df['opl'], merged_df[m])
    print(f"{m}: rho={rho:.3f}, p={p:.4f}")

Correlation FRE and Jargon use (corresponding to thesis section "Correlation language complexity and jargon use")

In [ ]:
level_df

In [ ]:
bel_df = pd.read_csv("all_bel.csv")
bel_df

In [ ]:
level_df = pd.read_csv("Level_Analysis/All_Level_Scores.csv")
level_df

In [ ]:
merged_df = pd.merge(
    bel_df, 
    level_df, 
    how='left', 
    on=['filename', 'label'],
    suffixes=('_bel', '_level')
)

doc_merged = merged_df[merged_df["label"]=="Doctor"].dropna()
patient_merged = merged_df[merged_df["label"]=="Patient"].dropna()

In [ ]:
doc_merged[doc_merged.isna().any(axis=1)]

In [ ]:
rho, p = spearmanr(doc_merged['fres'], doc_merged["Percentage_BEL_mentions"])
print(f"rho={rho:.3f}, p={p:.4f}")

In [ ]:
rho, p = spearmanr(patient_merged['fres'], patient_merged["Percentage_BEL_mentions"])
print(f"rho={rho:.3f}, p={p:.4f}")